In [25]:
import torch
import torch.nn as nn
import os
import time

In [2]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(4, 3)
        self.fc2 = nn.Linear(3, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x

model = MyModel()
state_dict = {
    "fc1.weight": torch.tensor([
        [0.1, 0.2, 0.3, 0.1],
        [0.2, 0.3, 0.1, 0.2],
        [0.3, 0.1, 0.2, 0.3],
    ]),

    "fc1.bias": torch.tensor([
        0.1, 0.2, 0.3
    ]),

    "fc2.weight": torch.tensor([
        [0.1, 0.2, 0.3],
        [0.2, 0.3, 0.1],
    ]),

    "fc2.bias": torch.tensor([
        0.1, 0.2
    ]),
}

model.load_state_dict(state_dict)

<All keys matched successfully>

In [3]:
model.eval()

MyModel(
  (fc1): Linear(in_features=4, out_features=3, bias=True)
  (fc2): Linear(in_features=3, out_features=2, bias=True)
)

### Dynamic

In [10]:
quantized_model = torch.ao.quantization.quantize_dynamic(
    model,
    {nn.Linear},
    dtype=torch.qint8
)

/tmp/ipykernel_17461/2260267086.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.ao.quantization.quantize_dynamic(


In [12]:
quantized_model

MyModel(
  (fc1): DynamicQuantizedLinear(in_features=4, out_features=3, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
  (fc2): DynamicQuantizedLinear(in_features=3, out_features=2, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
)

In [15]:
quantized_model.fc1.weight()

tensor([[0.0988, 0.2000, 0.2988, 0.0988],
        [0.2000, 0.2988, 0.0988, 0.2000],
        [0.2988, 0.0988, 0.2000, 0.2988]], size=(3, 4), dtype=torch.qint8,
       quantization_scheme=torch.per_tensor_affine, scale=0.0023529413156211376,
       zero_point=0)

In [16]:
model.fc1.weight

Parameter containing:
tensor([[0.1000, 0.2000, 0.3000, 0.1000],
        [0.2000, 0.3000, 0.1000, 0.2000],
        [0.3000, 0.1000, 0.2000, 0.3000]])

In [22]:
round(0.1 / 0.0023529413156211376) * 0.0023529413156211376, quantized_model.fc1.weight()[0][0].item()

(0.09882353525608778, 0.09882353246212006)

In [13]:
x = torch.tensor([
    [1.0, 2.0, 3.0, 4.0]
])

with torch.no_grad():
    y_original = model(x)
    y_quantized = quantized_model(x)

print("Original:")
print(y_original)

print("Quantized:")
print(y_quantized)

Original:
tensor([[1.4900, 1.4700]])
Quantized:
tensor([[1.4837, 1.4650]])


### Big Model

In [23]:
def get_size_mb(path):
    size_bytes = os.path.getsize(path)
    return size_bytes / (1024 ** 2)
    
class MyModel(nn.Module):
    def __init__(self, n_layers=3, hidden_dim=1000):
        super().__init__()
        layers = []
        for _ in range(n_layers):
            layers.append(
                nn.Linear(hidden_dim, hidden_dim, bias=False)
            )
            layers.append(nn.ReLU())
        layers.append(
            nn.Linear(hidden_dim, 1, bias=False)
        )
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

In [56]:
model = MyModel(n_layers=100)
with torch.no_grad():
    for param in model.parameters():
        param.uniform_(-0.1, 0.1)
        
torch.save(model.state_dict(), "model.pth")
original_size = get_size_mb("model.pth")
print(f"Original size : {original_size:.2f} MB")

Original size : 381.50 MB


In [57]:
quantized_model = torch.ao.quantization.quantize_dynamic(
    model,
    {nn.Linear},
    dtype=torch.qint8
)

/tmp/ipykernel_17461/2260267086.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.ao.quantization.quantize_dynamic(


In [58]:
torch.save(quantized_model.state_dict(), "quantized_model.pth")

In [59]:
quantized_size = get_size_mb("quantized_model.pth")
print(f"Quantized Size : {quantized_size:.2f} MB")

Quantized Size : 95.47 MB


In [60]:
loaded_model = MyModel(n_layers=100)
loaded_model.load_state_dict(
    torch.load("model.pth", weights_only=True)
)
loaded_model.eval()
pass

In [61]:
loaded_quantized_model = torch.ao.quantization.quantize_dynamic(
    MyModel(n_layers=100),
    {nn.Linear},
    dtype=torch.qint8
)
loaded_quantized_model.load_state_dict(
    torch.load("quantized_model.pth", weights_only=True)
)
loaded_quantized_model.eval()
pass

/tmp/ipykernel_17461/1048684667.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  loaded_quantized_model = torch.ao.quantization.quantize_dynamic(


In [62]:
x = torch.rand(1, 1000)

with torch.no_grad():
    y_original = loaded_model(x)
    y_quantized = loaded_quantized_model(x)

print("Original:")
print(y_original)

print("Quantized:")
print(y_quantized)

Original:
tensor([[-1.2497e+11]])
Quantized:
tensor([[-1.2464e+11]])
